## Notebook to start a spark instance in an ArcGIS Pro Session.

### THE BELOW HAS TO BE THE FIRST IMPORT !

In [ ]:
from spark_esri import spark_start, spark_stop

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [ ]:
import os

import arcpy
import geofunctions as S

### Check if the jar exists

In [ ]:
spark_jars = os.path.expanduser("~/geofunctions-0.XX.jar")
os.path.exists(spark_jars)

In [ ]:
config = {
    "spark.driver.memory": "32G",
    "spark.executor.memory": "32G",
    # Adjust to your environment.
    "spark.jars": spark_jars,
}
spark = spark_start(config=config)

In [ ]:
spark.version

### Sample usage. Create random points in the current map extent and place them on the Map.

In [ ]:
project = arcpy.mp.ArcGISProject("CURRENT")

extent = project.activeView.camera.getExtent()
xmin = extent.XMin
ymin = extent.YMin
dx = extent.XMax - extent.XMin
dy = extent.YMax - extent.YMin

In [ ]:
(
    spark
    # Create 1000 points.
    .range(1000)
    # Get random x values
    .withColumn("x", F.rand() * dx + xmin)
    # Get random y values.
    .withColumn("y", F.rand() * dy + ymin)
    # Create a point geometry from implicit x/y.
    .withColumn("geom", S.st_point())
    # Convert the spark dataframe to a feature class to be displayed in Pro TOC.
    .to_feature_class(
        "MyPoints",
        sp_ref=-1,  # Use current map SR.
    )
)